In [1]:
# 导入模块
import sys
import pandas as pd
from pathlib import Path
sys.path.append("..")
from utils.set_fonts import apply_times_with_cn
en_fp, cn_fp = apply_times_with_cn(cn_font_path=r"C:\Windows\\Fonts\\msyh.ttc")

Applied font.family: ['Times New Roman', 'Microsoft YaHei', 'DejaVu Sans']
findfont Times -> C:\Windows\Fonts\times.ttf
findfont CN -> C:\Windows\\Fonts\\msyh.ttc


In [2]:
from utils.common import (DATASETS, SEEDS, load_json, set_seed, load_data_and_info, 
                          get_per_fold_result, get_train_and_test_pos, get_gens)
from experiment.runner import run_all
from experiment.ifc_pipeline import make_ifc_cell

dataset_name = "breast_cancer_coimbra"
df, X, y, num_cols, cat_cols, target, datasets_info = load_data_and_info(dataset_name)

[KeOps] Warning : CUDA libraries not found or could not be loaded; Switching to CPU only.


In [3]:
## [1] 执行运行主程序

overrides = dict(token_dim=16, layer1=128, kl_warmup_steps=50, use_uncertainty_weighting=True,)
ft_overrides = dict(learning_rate=1e-4, patience1=20, epochs=500)

cells = [
    make_ifc_cell("none", False, overrides=overrides, ft_overrides=ft_overrides),
    make_ifc_cell("none", True, overrides=overrides, ft_overrides=ft_overrides),
    make_ifc_cell("bootstrap", False, overrides=overrides, ft_overrides=ft_overrides),
    make_ifc_cell("bootstrap", True, overrides=overrides, ft_overrides=ft_overrides),
    make_ifc_cell("smote", False, overrides=overrides, ft_overrides=ft_overrides),
    make_ifc_cell("smote", True, overrides=overrides, ft_overrides=ft_overrides),
]
 
run_all(datasets=['TCGA_InfoWithGrade'], ifc_cells= cells, resume=False)

Training Progress:  95%|█████████▍| 474/500 [00:14<00:00, 32.94it/s]


早停触发(PQ准则, alpha=1.0)：epoch=475, gen_loss=25.88, progress=0.00


Training Progress:  44%|████▍     | 221/500 [00:12<00:15, 18.40it/s]


早停触发(PQ准则, alpha=1.0)：epoch=222, gen_loss=10.98, progress=0.00


Training Progress:  32%|███▏      | 162/500 [00:05<00:10, 32.18it/s]


...


Training Progress:  96%|█████████▋| 482/500 [00:15<00:00, 32.08it/s]


早停触发(PQ准则, alpha=1.0)：epoch=483, gen_loss=27.58, progress=0.00


Training Progress:  60%|██████    | 300/500 [00:09<00:06, 31.33it/s]


早停触发(PQ准则, alpha=1.0)：epoch=301, gen_loss=18.85, progress=15.35


Training Progress: 100%|██████████| 500/500 [00:15<00:00, 32.19it/s]


In [5]:
run_all(datasets=['Thyroid_Diff','diabetes_risk_prediction', 'framingham', 'hepatitis'], 
        ifc_cells= cells, resume=True)

[skip] Thyroid_Diff seed=42 fold=1
[skip] Thyroid_Diff seed=42 fold=2
[skip] Thyroid_Diff seed=42 fold=3
[skip] Thyroid_Diff seed=42 fold=4
[skip] Thyroid_Diff seed=42 fold=5
[skip] Thyroid_Diff seed=456 fold=1
[skip] Thyroid_Diff seed=456 fold=2
[skip] Thyroid_Diff seed=456 fold=3
[skip] Thyroid_Diff seed=456 fold=4
[skip] Thyroid_Diff seed=456 fold=5
[skip] Thyroid_Diff seed=1024 fold=1
[skip] Thyroid_Diff seed=1024 fold=2
[skip] Thyroid_Diff seed=1024 fold=3
[skip] Thyroid_Diff seed=1024 fold=4
[skip] Thyroid_Diff seed=1024 fold=5
[skip] Thyroid_Diff seed=2026 fold=1
[skip] Thyroid_Diff seed=2026 fold=2
[skip] Thyroid_Diff seed=2026 fold=3
[skip] Thyroid_Diff seed=2026 fold=4
[skip] Thyroid_Diff seed=2026 fold=5
[skip] Thyroid_Diff seed=3407 fold=1
[skip] Thyroid_Diff seed=3407 fold=2
[skip] Thyroid_Diff seed=3407 fold=3
[skip] Thyroid_Diff seed=3407 fold=4
[skip] Thyroid_Diff seed=3407 fold=5
[skip] diabetes_risk_prediction seed=42 fold=1
[skip] diabetes_risk_prediction seed=42 fol

In [ ]:
cls_results = get_per_fold_result(dataset_name, 42, 1)
cls_results

In [ ]:
def get_fidelity_gens(dataset_name, seed, fold, method_tag):
    ROOT = Path().cwd().parent
    path = ROOT / 'artifacts'/ dataset_name / f'seed{seed}' / f'fold{fold}'/ method_tag / 'final.parquet'
    return pd.read_parquet(path)

X_syn = get_fidelity_gens(dataset_name, 42, 1, "none_noft")
X_syn[num_cols].describe().round(2)

In [ ]:
train_pos, test_pos = get_train_and_test_pos(dataset_name, 42, 1)